# PAQFE-v2 vs Standard QML — Full Repeated Composition-Level GroupKFold

**Purpose:** Replace the preliminary single-repeat experiment with a full **5 repeats × 5 folds** unseen-composition comparison.

- Dataset: same Excel workbook, worksheet `B_Model_Ready`
- Expected cleaned data: 49 observations and 14 unique compositions
- Models: Standard QML vs PAQFE-v2
- Seeds: 42, 43, 44, 45, 46
- Folds: 5 per repeat; same split used for both models
- Training: 150 epochs per model/fold
- Preprocessing: feature and target scalers fitted on the training fold only
- Outputs: fold metrics, out-of-fold predictions, pooled metrics per repeat, summaries, and a ZIP of results
- Persistent checkpoints: saves each completed model-fold to Google Drive and resumes by skipping completed runs

**Important:** This notebook preserves the preliminary experiment's model settings. Do not tune using held-out folds during this evaluation.

**Recovery:** If Colab disconnects, reconnect, rerun the notebook from the beginning, mount the same Google Drive, upload the same workbook, rerun the data/model/split setup cells, and rerun the training cell. Completed model-folds are skipped. The model-fold that was interrupted may need to run again. Do not change the dataset or settings when resuming.

## 0. Install dependencies
Run this cell first. `GroupKFold(shuffle=True, random_state=...)` requires a recent scikit-learn version.

In [ ]:
%pip install -q pennylane openpyxl "scikit-learn>=1.6"

## 1. Imports and fixed experimental settings

In [ ]:
import os
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

SEEDS = [42, 43, 44, 45, 46]
N_GROUP_SPLITS = 5
N_EPOCHS_GROUPED = 150
N_QUBITS = 10
N_LAYERS = 3
LR_VQC = 0.02
LR_ALPHA = 0.005
LAMBDA_PRIOR = 0.01
TARGET = "UTS_MPa"

STANDARD_FEATURE_ORDER = [
    "xAl", "VEC", "dHmix", "dSmix", "delta",
    "Omega", "dchi", "Mean_Tm", "Temperature_C", "log10_Strain_rate"
]
GROUPED_FEATURE_ORDER = [
    "xAl", "VEC", "dchi", "dHmix", "dSmix",
    "Omega", "Mean_Tm", "delta", "Temperature_C", "log10_Strain_rate"
]
COMPOSITION_COLS = ["Al_norm", "Co_norm", "Cr_norm", "Fe_norm", "Ni_norm"]
STRONG_RELATIONSHIP_DESCRIPTORS = {"delta", "Omega", "xAl", "VEC"}
UPWEIGHT_FACTOR = 1.2

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Seeds:", SEEDS)
print(f"Planned model-fold training runs: {len(SEEDS) * N_GROUP_SPLITS * 2}")

## 1A. Mount Google Drive for persistent checkpoints

Checkpoints are written after every completed model-fold. This storage survives a Colab runtime reset, unlike `/content`. Use the same Google account and keep the checkpoint folder intact when resuming.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_ROOT = "/content/drive/MyDrive/PAQFE_v2_GroupKFold_Checkpoints"
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print("Persistent checkpoint folder:", CHECKPOINT_ROOT)

## 2. Upload and validate the same model-ready dataset

Upload the **same Excel workbook** used for the preliminary run. The notebook reads `B_Model_Ready` and constructs composition groups from the five normalized elemental fractions rounded to three decimal places.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded.")
file_path = list(uploaded.keys())[0]

df = pd.read_excel(file_path, sheet_name="B_Model_Ready")

required = list(dict.fromkeys(
    STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS
))
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[required] = df[required].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=required).copy()
df = df.reset_index(drop=True)

df["Composition_Group"] = (
    df[COMPOSITION_COLS].round(3).astype(str).agg("_".join, axis=1)
)

print("Dataset:", os.path.basename(file_path))
print("Cleaned observations:", len(df))
print("Unique compositions:", df["Composition_Group"].nunique())
print("Target:", TARGET)
display(df[[TARGET, "Composition_Group"]].head())

if len(df) != 49 or df["Composition_Group"].nunique() != 14:
    print(
        "WARNING: Counts differ from the preliminary report (49 observations, "
        "14 compositions). Check workbook version, column values, and missing-data handling."
    )

## 3. Physics prior and PAQFE-v2 entanglement

In [ ]:
physics_prior = torch.tensor(
    [
        UPWEIGHT_FACTOR if feature in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0
        for feature in GROUPED_FEATURE_ORDER
    ],
    dtype=torch.float64
)

QUBIT_BLOCKS = {
    "composition": [0, 1, 2],
    "thermodynamic": [3, 4, 5, 6],
    "structural": [7],
    "context": [8, 9],
}

def ring_entangle(wires):
    if len(wires) < 2:
        return
    for i in range(len(wires)):
        qml.CNOT(wires=[wires[i], wires[(i + 1) % len(wires)]])

def physics_entangling_layer():
    ring_entangle(QUBIT_BLOCKS["composition"])
    ring_entangle(QUBIT_BLOCKS["thermodynamic"])
    ring_entangle(QUBIT_BLOCKS["context"])
    fmap = {feature: i for i, feature in enumerate(GROUPED_FEATURE_ORDER)}
    qml.CNOT(wires=[fmap["delta"], fmap["Omega"]])
    qml.CNOT(wires=[fmap["xAl"], fmap["VEC"]])

print("Physics prior:", physics_prior.tolist())

## 4. Model builders
The Standard QML and PAQFE-v2 models share the 10-qubit, 3-layer variational backbone and linear readout. PAQFE-v2 adds trainable physics scaling and physics-aware entanglement.

In [ ]:
def build_standard_qml():
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(
            inputs, wires=range(N_QUBITS), rotation="Y"
        )
        qml.templates.StronglyEntanglingLayers(
            weights, wires=range(N_QUBITS)
        )
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

    weight_shape = qml.StronglyEntanglingLayers.shape(
        n_layers=N_LAYERS, n_wires=N_QUBITS
    )

    class StandardQML(nn.Module):
        def __init__(self):
            super().__init__()
            self.weights = nn.Parameter(
                0.01 * torch.randn(weight_shape, dtype=torch.float64)
            )
            self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)

        def forward(self, x):
            q = torch.stack(
                [
                    torch.stack(circuit(sample, self.weights))
                    for sample in x
                ],
                dim=0,
            )
            return self.head(q).squeeze(-1)

    return StandardQML()


def build_paqfe_v2():
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(
            inputs, wires=range(N_QUBITS), rotation="Y"
        )
        physics_entangling_layer()
        qml.templates.StronglyEntanglingLayers(
            weights, wires=range(N_QUBITS)
        )
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

    weight_shape = qml.StronglyEntanglingLayers.shape(
        n_layers=N_LAYERS, n_wires=N_QUBITS
    )

    class PAQFEv2(nn.Module):
        def __init__(self):
            super().__init__()
            self.alpha = nn.Parameter(physics_prior.clone())
            self.weights = nn.Parameter(
                0.01 * torch.randn(weight_shape, dtype=torch.float64)
            )
            self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)

        def forward(self, x):
            theta = x * self.alpha
            q = torch.stack(
                [
                    torch.stack(circuit(sample, self.weights))
                    for sample in theta
                ],
                dim=0,
            )
            return self.head(q).squeeze(-1)

    return PAQFEv2()

print("Model builders defined.")

## 5. Smoke test before the full experiment

This tests the forward pass and gradient flow. If this cell fails, **do not start the 50 training runs**; resolve the error first.

In [ ]:
def smoke_test_model(model):
    model.train()
    x = torch.randn(3, N_QUBITS, dtype=torch.float64)
    y = torch.randn(3, dtype=torch.float64)
    pred = model(x)
    assert pred.shape == (3,), f"Unexpected output shape: {pred.shape}"
    loss = nn.MSELoss()(pred, y)
    loss.backward()
    assert all(
        p.grad is not None for p in model.parameters() if p.requires_grad
    ), "At least one trainable parameter has no gradient."

torch.manual_seed(42)
smoke_test_model(build_standard_qml())
torch.manual_seed(42)
smoke_test_model(build_paqfe_v2())
print("Both models passed forward-pass and gradient smoke tests.")

## 6. Validate all 25 grouped splits

Every composition must be entirely in either the training or test partition for each fold. Both models will use exactly the same split indices.

In [ ]:
groups = df["Composition_Group"].to_numpy()
X_for_split = df[STANDARD_FEATURE_ORDER]
y_for_split = df[TARGET]

split_cache = {}
for seed in SEEDS:
    splitter = GroupKFold(
        n_splits=N_GROUP_SPLITS,
        shuffle=True,
        random_state=seed,
    )
    splits = list(splitter.split(X_for_split, y_for_split, groups))
    assert len(splits) == N_GROUP_SPLITS

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])
        assert train_groups.isdisjoint(test_groups), (
            f"Composition leakage: seed={seed}, fold={fold}"
        )
        split_cache[(seed, fold)] = (train_idx, test_idx)

print(f"All {len(split_cache)} splits passed composition-leakage checks.")

## 7. Train one model on one fold

Important controls:
- Feature and target scalers are fitted on training rows only.
- Both models receive the same train/test indices within a fold.
- Each model/fold is trained once.
- Seeds are reset reproducibly before each model fit.

In [ ]:
def run_fold(model_name, train_idx, test_idx, seed):
    if model_name == "Standard QML":
        order = STANDARD_FEATURE_ORDER
    elif model_name == "PAQFE-v2":
        order = GROUPED_FEATURE_ORDER
    else:
        raise ValueError(f"Unknown model: {model_name}")

    X_train_raw = df.iloc[train_idx][order].to_numpy(dtype=np.float64)
    X_test_raw = df.iloc[test_idx][order].to_numpy(dtype=np.float64)
    y_train_raw = df.iloc[train_idx][TARGET].to_numpy(dtype=np.float64)
    y_test_raw = df.iloc[test_idx][TARGET].to_numpy(dtype=np.float64)

    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(X_train_raw)
    X_test = x_scaler.transform(X_test_raw)

    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()

    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    if model_name == "Standard QML":
        model = build_standard_qml()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR_VQC)
    else:
        model = build_paqfe_v2()
        optimizer = torch.optim.Adam(
            [
                {"params": [model.alpha], "lr": LR_ALPHA},
                {"params": [model.weights], "lr": LR_VQC},
                {"params": model.head.parameters(), "lr": LR_VQC},
            ]
        )

    X_train_t = torch.tensor(X_train, dtype=torch.float64)
    y_train_t = torch.tensor(y_train, dtype=torch.float64)
    X_test_t = torch.tensor(X_test, dtype=torch.float64)

    loss_fn = nn.MSELoss()
    model.train()

    for epoch in range(N_EPOCHS_GROUPED):
        optimizer.zero_grad()
        pred_train = model(X_train_t)
        loss = loss_fn(pred_train, y_train_t)

        if model_name == "PAQFE-v2":
            prior_penalty = torch.mean((model.alpha - physics_prior) ** 2)
            loss = loss + LAMBDA_PRIOR * prior_penalty

        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_standardized = model(X_test_t).cpu().numpy()

    y_pred = y_scaler.inverse_transform(
        pred_standardized.reshape(-1, 1)
    ).ravel()

    return y_test_raw, y_pred

## 8. Run or resume the full 5-repeat × 5-fold experiment

The notebook performs 50 model-fold runs. After each model-fold finishes, it saves fold metrics and out-of-fold predictions to Google Drive. If the runtime disconnects, completed model-folds are skipped on the next run.

**Recovery steps:** reconnect to Colab; rerun cells from the beginning; mount the same Google Drive account; upload the same Excel workbook; rerun the data, model, and split cells; then rerun this training cell. The currently interrupted model-fold may need to run again. Do not change experiment settings when resuming.

In [ ]:
import hashlib
import json
from pathlib import Path

# Fingerprint the cleaned dataset and settings so checkpoints from a different
# workbook/configuration cannot be silently mixed into this experiment.
fingerprint_cols = list(dict.fromkeys(
    STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS + ["Composition_Group"]
))
data_fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(df[fingerprint_cols], index=True).values.tobytes()
).hexdigest()[:16]

experiment_config = {
    "data_fingerprint": data_fingerprint,
    "sheet": "B_Model_Ready",
    "seeds": SEEDS,
    "n_group_splits": N_GROUP_SPLITS,
    "epochs": N_EPOCHS_GROUPED,
    "n_qubits": N_QUBITS,
    "n_layers": N_LAYERS,
    "lr_vqc": LR_VQC,
    "lr_alpha": LR_ALPHA,
    "lambda_prior": LAMBDA_PRIOR,
    "target": TARGET,
    "standard_feature_order": STANDARD_FEATURE_ORDER,
    "grouped_feature_order": GROUPED_FEATURE_ORDER,
    "composition_cols": COMPOSITION_COLS,
    "upweight_factor": UPWEIGHT_FACTOR,
}
config_hash = hashlib.sha256(
    json.dumps(experiment_config, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

CHECKPOINT_DIR = Path(CHECKPOINT_ROOT) / f"groupkfold_{config_hash}"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CHECKPOINT_DIR / "experiment_config.json"
FOLD_CSV = CHECKPOINT_DIR / "checkpoint_fold_results.csv"
OOF_CSV = CHECKPOINT_DIR / "checkpoint_oof_predictions.csv"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        previous_config = json.load(f)
    if previous_config != experiment_config:
        raise RuntimeError(
            "Checkpoint settings/dataset do not match this notebook run. "
            "Do not combine these checkpoints with the current experiment."
        )
else:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(experiment_config, f, indent=2)

def atomic_csv_save(frame, destination):
    """Write to a temporary file first, then replace the previous checkpoint."""
    temporary = str(destination) + ".tmp"
    frame.to_csv(temporary, index=False)
    os.replace(temporary, destination)

if FOLD_CSV.exists():
    records = pd.read_csv(FOLD_CSV).to_dict("records")
else:
    records = []

if OOF_CSV.exists():
    oof_rows = pd.read_csv(OOF_CSV).to_dict("records")
else:
    oof_rows = []

# Remove any duplicate prediction rows that could result if a disconnect occurred
# between writing the prediction checkpoint and the fold-results checkpoint.
if oof_rows:
    oof_frame = pd.DataFrame(oof_rows).drop_duplicates(
        subset=["Seed", "Fold", "Model", "Row_Index"], keep="last"
    )
    oof_rows = oof_frame.to_dict("records")

def checkpoint_key(row):
    return (int(row["Seed"]), int(row["Fold"]), str(row["Model"]))

completed = {checkpoint_key(row) for row in records}
expected_runs = len(SEEDS) * N_GROUP_SPLITS * 2

print("Checkpoint directory:", CHECKPOINT_DIR)
print(f"Already completed: {len(completed)}/{expected_runs} model-fold runs")
print("Only missing model-fold runs will execute.")

for repeat, seed in enumerate(SEEDS, start=1):
    for fold in range(1, N_GROUP_SPLITS + 1):
        train_idx, test_idx = split_cache[(seed, fold)]
        test_original_indices = df.index.to_numpy()[test_idx]

        for model_name in ["Standard QML", "PAQFE-v2"]:
            key = (seed, fold, model_name)
            if key in completed:
                print(f"SKIP checkpointed: Repeat {repeat}/5 | Fold {fold}/5 | {model_name}")
                continue

            y_true, y_pred = run_fold(model_name, train_idx, test_idx, seed)

            fold_mae = mean_absolute_error(y_true, y_pred)
            fold_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            fold_r2 = r2_score(y_true, y_pred)

            records.append({
                "Repeat": repeat,
                "Seed": seed,
                "Fold": fold,
                "Model": model_name,
                "MAE": fold_mae,
                "RMSE": fold_rmse,
                "R2": fold_r2,
                "N_Test": len(test_idx),
            })

            for idx, actual, predicted in zip(test_original_indices, y_true, y_pred):
                oof_rows.append({
                    "Repeat": repeat,
                    "Seed": seed,
                    "Fold": fold,
                    "Model": model_name,
                    "Row_Index": int(idx),
                    "Composition_Group": str(df.iloc[idx]["Composition_Group"]),
                    "y_true_UTS_MPa": float(actual),
                    "y_pred_UTS_MPa": float(predicted),
                    "Residual_Actual_minus_Predicted": float(actual - predicted),
                })

            # Deduplicate predictions defensively, then persist results immediately.
            oof_frame = pd.DataFrame(oof_rows).drop_duplicates(
                subset=["Seed", "Fold", "Model", "Row_Index"], keep="last"
            )
            oof_rows = oof_frame.to_dict("records")
            atomic_csv_save(oof_frame, OOF_CSV)
            atomic_csv_save(pd.DataFrame(records), FOLD_CSV)
            completed.add(key)

            print(
                f"SAVED to Drive | Repeat {repeat}/5 | Fold {fold}/5 | {model_name} | "
                f"MAE={fold_mae:.2f} | RMSE={fold_rmse:.2f} | R2={fold_r2:.3f}"
            )
            print(f"Progress: {len(completed)}/{expected_runs} model-fold runs complete")

fold_results = pd.DataFrame(records).sort_values(
    ["Repeat", "Fold", "Model"]
).reset_index(drop=True)
oof_predictions = pd.DataFrame(oof_rows).drop_duplicates(
    subset=["Seed", "Fold", "Model", "Row_Index"], keep="last"
).sort_values(["Repeat", "Fold", "Model", "Row_Index"]).reset_index(drop=True)

if len(fold_results) != expected_runs:
    print(f"WARNING: only {len(fold_results)}/{expected_runs} model-fold runs are complete.")
else:
    print("All model-fold runs completed.")

display(fold_results.head(10))
print("Fold result rows:", len(fold_results))
print("Out-of-fold prediction rows:", len(oof_predictions))

## 9. Fold-level summary

This summarizes the 25 fold scores per model. It is useful for describing fold-to-fold variation, but it is **not the primary pooled-repeat estimate**.

In [ ]:
fold_summary = (
    fold_results.groupby("Model")[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"])
)
display(fold_summary)

## 10. Pooled out-of-fold metrics for each repeat

Within each repeat, combine the held-out predictions from all five folds and calculate metrics once. This yields five pooled evaluation results per model, each covering all 49 observations exactly once in that repeat.

Use these pooled-repeat results as the main comparison. Fold-level R² can be unstable because some test folds are small.

In [ ]:
pooled_rows = []

for (repeat, seed, model_name), part in oof_predictions.groupby(
    ["Repeat", "Seed", "Model"], sort=True
):
    y_true = part["y_true_UTS_MPa"].to_numpy()
    y_pred = part["y_pred_UTS_MPa"].to_numpy()

    pooled_rows.append({
        "Repeat": int(repeat),
        "Seed": int(seed),
        "Model": model_name,
        "N_Predictions": len(part),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    })

pooled_repeat_results = pd.DataFrame(pooled_rows)
display(pooled_repeat_results)

## 11. Main presentation-ready summary
Mean ± sample standard deviation across the five pooled-repeat results. These are repeat-to-repeat validation summaries, not independent-dataset confidence intervals.

In [ ]:
pooled_summary = (
    pooled_repeat_results.groupby("Model")[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"])
)
display(pooled_summary)

print("\nPRESENTATION-READY POOLED-REPEAT SUMMARY")
for model_name in ["Standard QML", "PAQFE-v2"]:
    row = pooled_summary.loc[model_name]
    print(
        f"{model_name}: "
        f"MAE {row[('MAE','mean')]:.2f} ± {row[('MAE','std')]:.2f} MPa; "
        f"RMSE {row[('RMSE','mean')]:.2f} ± {row[('RMSE','std')]:.2f} MPa; "
        f"R² {row[('R2','mean')]:.3f} ± {row[('R2','std')]:.3f}"
    )

## 12. Save and download all results

Save the notebook after execution as well. The CSV files preserve the numerical evidence needed for plots, paper tables, residual analysis, and later explainability work.

In [ ]:
OUTPUT_FILES = {
    "PAQFE_v2_GroupKFold_fold_results.csv": fold_results,
    "PAQFE_v2_GroupKFold_oof_predictions.csv": oof_predictions,
    "PAQFE_v2_GroupKFold_fold_summary.csv": fold_summary,
    "PAQFE_v2_GroupKFold_pooled_repeat_results.csv": pooled_repeat_results,
    "PAQFE_v2_GroupKFold_pooled_summary.csv": pooled_summary,
}

for filename, table in OUTPUT_FILES.items():
    table.to_csv(filename, index=True if "summary" in filename else False)
    table.to_csv(str(CHECKPOINT_DIR / filename), index=True if "summary" in filename else False)

zip_name = "PAQFE_v2_Full_GroupKFold_Results.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in OUTPUT_FILES:
        zf.write(filename)
        zf.write(str(CHECKPOINT_DIR / filename), arcname="drive_" + filename)

print("Persistent checkpoints and exports saved in:", CHECKPOINT_DIR)
print("Saved files:")
for filename in list(OUTPUT_FILES) + [zip_name]:
    print(" -", filename)

from google.colab import files
files.download(zip_name)

## 13. Final integrity checklist

Before reporting results, confirm:
- [ ] The same workbook and `B_Model_Ready` sheet were used.
- [ ] Cleaned observation/composition counts were checked.
- [ ] All 25 splits passed the no-composition-leakage check.
- [ ] Both models used identical fold indices.
- [ ] Scalers were fitted on training data only.
- [ ] The smoke test passed for both models.
- [ ] Exactly 50 model-fold results were produced.
- [ ] Exactly 490 out-of-fold prediction rows were produced if the dataset has 49 observations (49 × 5 repeats × 2 models).
- [ ] The five CSV files and executed notebook were saved.
- [ ] The presentation clearly labels this as repeated grouped validation and does not overstate the small dataset's certainty.

**Interpretation caution:** With only 14 unique compositions, grouped folds are small and metrics may vary substantially. Report results honestly, even if PAQFE-v2 does not outperform Standard QML.
- [ ] Confirm checkpoints are present in Google Drive before closing the session.
- [ ] If resuming, use the same Google account, workbook, and notebook settings.